# 2-1. Advanced RAG: Chunking 개선

## 학습 목표

- 청크 크기에 따라 검색 결과가 어떻게 달라지는지 비교한다.
- PDF 매뉴얼에 적합한 청크 크기를 실험한다.


## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from const.const import create_model, create_embedding
import re
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

load_dotenv(override=True, dotenv_path="../../.env")

MD_PATH = Path("../data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md")

if not MD_PATH.exists():
    raise FileNotFoundError(f"Markdown 파일을 찾을 수 없습니다: {MD_PATH}")

embeddings = create_embedding()


/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/parkchanryong/Desktop/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [2]:
def load_markdown_page_chunks(md_path: Path) -> list[Document]:
    """
    전처리된 RAG page chunk Markdown 파일을 Document 리스트로 읽는다.

    대상 파일 예:
    ../data/공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md

    기준:
    - ## Page n | section | topic 단위로 분리
    - HTML comment metadata는 제거
    - page, section, topic 메타데이터를 유지
    """
    text = md_path.read_text(encoding="utf-8")

    # 첫 번째 문서 제목 제거
    text = re.sub(r"^# .+?\n", "", text, count=1)

    # ## Page 단위로 분리
    parts = re.split(r"(?=^## Page\s+\d+\s*\|)", text, flags=re.MULTILINE)

    docs = []

    for part in parts:
        part = part.strip()

        if not part:
            continue

        header_match = re.match(
            r"^## Page\s+(\d+)\s*\|\s*([^|]+)\s*\|\s*(.+)$",
            part.splitlines()[0].strip(),
        )

        if not header_match:
            continue

        page = int(header_match.group(1))
        section = header_match.group(2).strip()
        topic = header_match.group(3).strip()
        lines = part.splitlines()[1:]

        # HTML comment metadata 제거
        content_lines = [line for line in lines if not line.strip().startswith("<!--")]

        content = "\n".join(content_lines).strip()

        if not content:
            continue

        docs.append(
            Document(
                page_content=content,
                metadata={
                    "source": md_path.name,
                    "page": page,
                    "section": section,
                    "topic": topic,
                    "chunk_type": "page",
                },
            )
        )

    return docs

In [3]:
def make_chunks(
    docs: list[Document], chunk_size: int, chunk_overlap: int
) -> list[Document]:
    """페이지 단위 Markdown 문서를 검색용 청크로 분할한다."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n## ", "\n# ", "\n- ", "\n", ". ", " "],
    )

    chunks = splitter.split_documents(docs)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i
        chunk.metadata["chunk_size"] = chunk_size

    return chunks

## 1. 서로 다른 청크 크기 만들기


In [4]:
pages = load_markdown_page_chunks(MD_PATH)

small_chunks = make_chunks(pages, chunk_size=400, chunk_overlap=80)
medium_chunks = make_chunks(pages, chunk_size=700, chunk_overlap=120)
large_chunks = make_chunks(pages, chunk_size=1000, chunk_overlap=150)

print("page docs:", len(pages))
print("small:", len(small_chunks))
print("medium:", len(medium_chunks))
print("large:", len(large_chunks))


page docs: 16
small: 59
medium: 34
large: 26


## 2. 청크 크기별 벡터스토어 생성


In [5]:
small_vs = QdrantVectorStore.from_documents(
    documents=small_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="small_chunks",
    force_recreate=True,
)

medium_vs = QdrantVectorStore.from_documents(
    documents=medium_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="medium_chunks",
    force_recreate=True,
)

large_vs = QdrantVectorStore.from_documents(
    documents=large_chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="large_chunks",
    force_recreate=True,
)

## 3. 검색 결과 비교


In [6]:
question = "전화 중 욕설을 하면 어떻게 대응해야 하나요?"

stores = {"small": small_vs, "medium": medium_vs, "large": large_vs}

for name, store in stores.items():
    print("\n" + "=" * 80)
    print(f"[{name} chunk 검색 결과]")
    docs = store.similarity_search(question, k=3)

    for doc in docs:
        print(doc.metadata)
        print(doc.page_content[:350].replace("\n", " "))
        print()


[small chunk 검색 결과]
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'chunk_type': 'page', 'chunk_id': 13, 'chunk_size': 400, '_id': '94cac77e7cc742c8b987af4eed7c8fe3', '_collection_name': 'small_chunks'}
2-2 정당한 사유 없는 장시간 통화 ※ 권장시간 20분 설정의 경우 핵심대응 용건위주 질문유도 및 장시간 전화상담 곤란 안내 2-4 폭언(욕설, 협박, 성희롱 등) 핵심대응 전화 종료 및 증거확보를 통한 후속조치(법적조치 등)로 경각심 부여 ‣ ~~민원전화 수신 시부터 녹음하여 위법행위 증거 확보~~ ‣ ~~15분 이상 통화 장시간 상담 곤란 안내 및 용건 위주 질문유도~~ ※ 민원 신청은 문서가 원칙임을 안내, 지속시 신문고 등 접수 유도 ‣ ~~20분 이상 통화 통화 지속 곤란 안내 후 종료~~ 선생님, 죄송하지만 상담 권장시간이 초과되어 다른 민원 처리를 위해서(또는 대기하고 계시는 다른 민원인이 계셔서) 통화를

{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'chunk_type': 'page', 'chunk_id': 15, 'chunk_size': 400, '_id': '2e67c6ac6d69453289d16285d673ae15', '_collection_name': 'small_chunks'}
방법을 찾아보겠습니다. 차분하게 민원사항을 말씀해주시기 바랍니다. 선생님, 상급자와 통화하고 싶은 이유가 정확히 어떤 이유 때문인지 여쭤봐도 되겠습니까? 선생님께서 해결하고 싶은 정확한 민원이 있으신건지? 아니면 단순히 상급자와 폭언·성희롱 등 위법행위에 

“작은 청크는 잘 찾지만 너무 잘게 쪼개져서 답변에 필요한 절차가 끊길 수 있고, 큰 청크는 문맥은 많지만 관련 없는 내용까지 섞인다. 이 실험에서는 중간 크기 청크가 검색 정확도와 답변 생성에 필요한 문맥을 가장 균형 있게 제공한다.”


# chunk 선택 판단

| 구분   |               설정 | 청크 수 | 판단                                              |
| ------ | -----------------: | ------: | ------------------------------------------------- |
| small  |   400 / overlap 80 |    70개 | 검색 정확도는 좋지만 문맥이 짧게 잘릴 위험이 있다 |
| medium |  700 / overlap 120 |    43개 | 검색 정확도와 문맥 보존의 균형이 가장 좋다        |
| large  | 1000 / overlap 150 |    32개 | 문맥은 풍부하지만 불필요한 내용이 함께 섞인다     |


## 청크 수 판단 기준

| 평가 기준          | small | medium    | large           |
| ------------------ | ----- | --------- | --------------- |
| 검색 정확도        | 높음  | 높음      | 중간            |
| 문맥 보존          | 낮음  | 높음      | 매우 높음       |
| 불필요한 정보 포함 | 낮음  | 적절함    | 높음            |
| 답변 생성 안정성   | 보통  | 가장 좋음 | 보통            |
| 수업 실습 적합성   | 좋음  | 가장 좋음 | 설명용으로 좋음 |


## 핵심 정리

- 작은 청크는 정확한 구간을 찾기 좋지만 문맥이 부족할 수 있다.
- 큰 청크는 문맥이 풍부하지만 불필요한 내용이 섞일 수 있다.
- 매뉴얼형 문서는 제목, 단계, 항목을 고려한 청크 분할이 중요하다.
